In [1]:
# 🚀 1. Inicialização do Spark
import os
import sys

os.environ['PYSPARK_PYTHON'] = "/Users/lpasch/sources/spark-3.4.3/python:/Users/lpasch/sources/spark-3.4.3/python/lib/py4j-0.10.9.7-src.zip:/Users/lpasch/sources/spark-3.4.3/python:/Users/lpasch/sources/spark-3.4.3/python/lib/py4j-0.10.9.7-src.zip:/Users/lpasch/sources/spark-3.4.3/python:/Users/lpasch/sources/spark-3.4.3/python/lib/py4j-0.10.9.7-src.zip:"
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['JAVA_HOME'] = "/opt/homebrew/Cellar/openjdk@11/11.0.31"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_unixtime, avg, count
from dotenv import load_dotenv
load_dotenv('.env')  # Carregar o arquivo .env

aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_region = "us-east-2"

spark = SparkSession.builder \
    .appName("Tesouro Silver layer") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/homebrew/anaconda3/envs/data-ai-env/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lpasch/.ivy2/cache
The jars for the packages stored in: /Users/lpasch/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-afe0f139-65b4-493d-8e83-75c87b89185b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (219ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (6499ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7

In [2]:
# -----------------------
# 3. Ler os Dados Brutos - Camada Bronze
# não se esqueça de alterar o nome do seu bucket
# -----------------------

bronze_path = "s3a://xp-project-dl-bronze/raw-data/kafka/tesouro_ipca"

# Ler os dados do S3 e testar a conexão
try:
    df_bronze = spark.read.json(bronze_path)
    print("Leitura bem-sucedida!")
    df_bronze.show()
except Exception as e:
    print(f"Erro ao acessar o S3: {e}")


26/06/21 18:19:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Leitura bem-sucedida!
+-----------+-------------+---------------+-----------+-------------+------------+----+----------+-------------+----+-----+---+----+
|CompraManha|    Data_Base|Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo|VendaManha|    dt_update|year|month|day|hour|
+-----------+-------------+---------------+-----------+-------------+------------+----+----------+-------------+----+-----+---+----+
|       7.97|1710720000000|  1723680000000|    4112.86|      4116.28|     4112.86|IPCA|      8.09|1782074045793|2026|    6| 21|  17|
|       5.72|1710720000000|  1786752000000|    3704.66|      3715.87|     3704.66|IPCA|      5.84|1782074045793|2026|    6| 21|  17|
|       5.63|1710720000000|  1873497600000|     3189.6|      3209.12|      3189.6|IPCA|      5.75|1782074045793|2026|    6| 21|  17|
|       5.72|1710720000000|  2062800000000|    2263.21|      2292.53|     2263.21|IPCA|      5.84|1782074045793|2026|    6| 21|  17|
|       5.84|1710720000000|  2378419200000|    

In [4]:
# -----------------------
# 4. Tratamento dos Dados - Camada Silver
# -----------------------
# Remover duplicações e converter timestamps para datas legíveis
df_silver = df_bronze.dropDuplicates()

# Tratar timestamps e converter para formato legível
df_silver = df_silver.withColumn("Data_Vencimento", from_unixtime(col("Data_Vencimento") / 1000, "yyyy-MM-dd")) \
                     .withColumn("Data_Base", from_unixtime(col("Data_Base") / 1000, "yyyy-MM-dd")) \
                     .withColumn("dt_update", from_unixtime(col("dt_update") / 1000, "yyyy-MM-dd HH:mm:ss"))

# Tratar valores nulos
df_silver = df_silver.fillna({
    "PUCompraManha": 0,
    "PUVendaManha": 0,
    "PUBaseManha": 0
})

# Visualizar os dados transformados
print("Dados Transformados (Silver):")
df_silver.show(truncate=False)

# Salvar os dados limpos no S3 em formato Parquet
silver_path = "s3a://xp-project-dl-silver/processed-data/ipca/silver/"
df_silver.write.mode("overwrite").parquet(silver_path)

Dados Transformados (Silver):


+-----------+----------+---------------+-----------+-------------+------------+----+----------+-------------------+----+-----+---+----+
|CompraManha|Data_Base |Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo|VendaManha|dt_update          |year|month|day|hour|
+-----------+----------+---------------+-----------+-------------+------------+----+----------+-------------------+----+-----+---+----+
|5.56       |2024-02-01|2035-05-14     |2263.16    |2293.51      |2263.16     |IPCA|5.68      |2026-06-21 17:34:05|2026|6    |21 |17  |
|5.93       |2023-01-17|2029-05-14     |2772.5     |2793.42      |2772.5      |IPCA|6.05      |2026-06-21 17:34:05|2026|6    |21 |17  |
|5.19       |2023-09-03|2029-05-14     |3085.46    |3106.34      |3085.46     |IPCA|5.31      |2026-06-21 17:34:05|2026|6    |21 |17  |
|5.53       |2022-02-15|2045-05-14     |1067.27    |1096.32      |1067.27     |IPCA|5.65      |2026-06-21 17:34:05|2026|6    |21 |17  |
|4.07       |2021-06-07|2045-05-14     |1330.99 

In [5]:
# -----------------------
# 5. Agregação e Métricas - Camada Gold
# -----------------------
# Calcular métricas agregadas
df_gold = df_silver.groupBy("Tipo").agg(
    avg("PUCompraManha").alias("Media_PUCompraManha"),
    avg("PUVendaManha").alias("Media_PUVendaManha"),
    count("*").alias("Total_Registros")
)

# Visualizar as métricas agregadas
print("Dados Agregados (Gold):")
df_gold.show(truncate=False)

# Salvar os dados agregados no S3 em formato Parquet
gold_path = "s3a://xp-project-dl-gold/analytics/ipca/gold/"
df_gold.write.mode("overwrite").parquet(gold_path)

Dados Agregados (Gold):


+-----+-------------------+------------------+---------------+
|Tipo |Media_PUCompraManha|Media_PUVendaManha|Total_Registros|
+-----+-------------------+------------------+---------------+
|IPCA |1831.3117034220545 |1816.4058071700192|36820          |
|IPCA+|1000.0             |1001.0            |1              |
+-----+-------------------+------------------+---------------+



In [ ]:
# -----------------------
# 6. Encerrar a Spark Session
# -----------------------
spark.stop()
